# 03 - Validación final SESCO

Esta notebook valida que el dataset unificado esté listo para alimentar el dashboard y genera archivos auxiliares para evitar recalcular reglas críticas en Streamlit.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# Resolver ruta raíz del repo buscando una carpeta que contenga /exploration.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "exploration").exists()), Path.cwd())

DATA_DIR = ROOT / "exploration" / "data" / "processed"
UNIFIED_PATH = DATA_DIR / "sesco_produccion_model_clean.csv"
RESUMEN_PATH = DATA_DIR / "sesco_validaciones_resumen.csv"

LATEST_EXPORT_PATH = DATA_DIR / "sesco_latest_periods_by_view.csv"
TOTALES_EXPORT_PATH = DATA_DIR / "sesco_totales_por_vista_resumen.csv"
CONFIG_EXPORT_PATH = DATA_DIR / "sesco_dashboard_config.csv"

required_paths = [UNIFIED_PATH, RESUMEN_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "No se encontraron archivos requeridos. "
        f"ROOT detectado: {ROOT}. Faltantes: {missing}"
    )

df = pd.read_csv(UNIFIED_PATH)
df_resumen = pd.read_csv(RESUMEN_PATH)

print(f"ROOT detectado: {ROOT}")
print(f"Dataset unificado: {UNIFIED_PATH}")
print(f"Resumen validaciones: {RESUMEN_PATH}")
print(f"Filas unificado: {len(df):,}")
print(f"Filas resumen: {len(df_resumen):,}")

ROOT detectado: c:\Proyectos\Dashboards\OilAndGas
Dataset unificado: c:\Proyectos\Dashboards\OilAndGas\exploration\data\processed\sesco_produccion_model_clean.csv
Resumen validaciones: c:\Proyectos\Dashboards\OilAndGas\exploration\data\processed\sesco_validaciones_resumen.csv
Filas unificado: 33,912
Filas resumen: 6


In [8]:
# 1) Validación de estructura del dataset unificado
print("=== Estructura general ===")
print(f"Cantidad de filas: {len(df):,}")
print(f"Cantidad de columnas: {df.shape[1]}")

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes.rename("dtype").to_frame())

print("\nNulos por columna:")
display(df.isna().sum().rename("nulos").sort_values(ascending=False).to_frame())

print("\nProductos disponibles:")
display(pd.DataFrame({"producto": sorted(df["producto"].dropna().unique())}))

print("Agrupador_tipo disponibles:")
display(pd.DataFrame({"agrupador_tipo": sorted(df["agrupador_tipo"].dropna().unique())}))

periodo_min_global = df["periodo_str"].min()
periodo_max_global = df["periodo_str"].max()
print(f"Rango general de períodos: {periodo_min_global} -> {periodo_max_global}")

print("\n=== Vista rápida del resumen existente ===")
display(df_resumen)

=== Estructura general ===
Cantidad de filas: 33,912
Cantidad de columnas: 10

Columnas disponibles:
['periodo_str', 'periodo_dt', 'anio', 'mes', 'producto', 'agrupador_tipo', 'agrupador_nombre', 'tipo_recurso', 'produccion', 'source_resource']

Tipos de datos:


,dtype
periodo_str,str
periodo_dt,str
anio,int64
mes,int64
producto,str
agrupador_tipo,str
agrupador_nombre,str
tipo_recurso,str
produccion,float64
source_resource,str



Nulos por columna:


,nulos
periodo_str,0
periodo_dt,0
anio,0
mes,0
producto,0
agrupador_tipo,0
agrupador_nombre,0
tipo_recurso,0
produccion,0
source_resource,0



Productos disponibles:


,producto
0,gas
1,petroleo


Agrupador_tipo disponibles:


,agrupador_tipo
0,cuenca
1,empresa
2,provincia


Rango general de períodos: 2009-01 -> 2026-04

=== Vista rápida del resumen existente ===


,resource_key,producto,agrupador_tipo,cantidad_filas_original,cantidad_filas_limpias,periodo_min,periodo_max_disponible,latest_valid_period,excluded_period,cantidad_agrupadores,cantidad_nulos_produccion,cantidad_produccion_negativa,observaciones
0,petroleo_provincia,petroleo,provincia,2505,2498,2009-01,2026-01,2025-12,2026-01,15,0,0,excluido periodo 2026-01
1,gas_provincia,gas,provincia,2493,2488,2009-01,2026-04,2026-03,2026-04,15,0,0,excluido periodo 2026-04
2,petroleo_cuenca,petroleo,cuenca,2013,2013,2009-01,2025-11,2025-11,NaN,17,0,0,NaN
3,gas_cuenca,gas,cuenca,2062,2059,2009-01,2026-05,2026-04,2026-05,17,0,0,excluido periodo 2026-05
4,petroleo_empresa,petroleo,empresa,12508,12507,2009-01,2026-05,2026-04,2026-05,127,0,1,excluido periodo 2026-05
5,gas_empresa,gas,empresa,12348,12347,2009-01,2026-05,2026-04,2026-05,127,0,0,excluido periodo 2026-05


In [9]:
# 2) Validación de duplicados con clave lógica
logical_key = [
    "periodo_str",
    "producto",
    "agrupador_tipo",
    "agrupador_nombre",
    "tipo_recurso",
]

dup_mask = df.duplicated(subset=logical_key, keep=False)
duplicados = df.loc[dup_mask].sort_values(logical_key).copy()

print("=== Duplicados por clave lógica ===")
print(f"Registros duplicados detectados: {len(duplicados):,}")
if duplicados.empty:
    print("No se encontraron duplicados para la clave lógica.")
else:
    display(duplicados.head(100))

=== Duplicados por clave lógica ===
Registros duplicados detectados: 0
No se encontraron duplicados para la clave lógica.


In [10]:
# 3) Validación de períodos por combinación producto + agrupador_tipo
periodos_por_vista = (
    df.groupby(["producto", "agrupador_tipo"], as_index=False)
    .agg(
        periodo_min=("periodo_str", "min"),
        periodo_max=("periodo_str", "max"),
        cantidad_periodos=("periodo_str", "nunique"),
        cantidad_filas=("periodo_str", "size"),
        cantidad_agrupadores=("agrupador_nombre", "nunique"),
    )
    .sort_values(["producto", "agrupador_tipo"])
    .reset_index(drop=True)
)

display(periodos_por_vista)

# Tabla auxiliar para Streamlit: último período válido por vista
latest_periods_by_view = (
    periodos_por_vista[["producto", "agrupador_tipo", "periodo_max"]]
    .rename(columns={"periodo_max": "latest_valid_period"})
)

latest_periods_by_view.to_csv(LATEST_EXPORT_PATH, index=False)
print(f"Exportado: {LATEST_EXPORT_PATH}")
display(latest_periods_by_view)

# Config de dashboard para no recalcular reglas críticas en frontend/streamlit.
dashboard_config = (
    periodos_por_vista[[
        "producto",
        "agrupador_tipo",
        "periodo_min",
        "periodo_max",
        "cantidad_agrupadores",
    ]]
    .rename(columns={"periodo_max": "latest_valid_period"})
    .sort_values(["producto", "agrupador_tipo"])
    .reset_index(drop=True)
)

dashboard_config.to_csv(CONFIG_EXPORT_PATH, index=False)
print(f"Exportado: {CONFIG_EXPORT_PATH}")
display(dashboard_config)

,producto,agrupador_tipo,periodo_min,periodo_max,cantidad_periodos,cantidad_filas,cantidad_agrupadores
0,gas,cuenca,2009-01,2026-04,208,2059,17
1,gas,empresa,2009-01,2026-04,208,12347,127
2,gas,provincia,2009-01,2026-03,207,2488,15
3,petroleo,cuenca,2009-01,2025-11,203,2013,17
4,petroleo,empresa,2009-01,2026-04,208,12507,127
5,petroleo,provincia,2009-01,2025-12,204,2498,15


Exportado: c:\Proyectos\Dashboards\OilAndGas\exploration\data\processed\sesco_latest_periods_by_view.csv


,producto,agrupador_tipo,latest_valid_period
0,gas,cuenca,2026-04
1,gas,empresa,2026-04
2,gas,provincia,2026-03
3,petroleo,cuenca,2025-11
4,petroleo,empresa,2026-04
5,petroleo,provincia,2025-12


Exportado: c:\Proyectos\Dashboards\OilAndGas\exploration\data\processed\sesco_dashboard_config.csv


,producto,agrupador_tipo,periodo_min,latest_valid_period,cantidad_agrupadores
0,gas,cuenca,2009-01,2026-04,17
1,gas,empresa,2009-01,2026-04,127
2,gas,provincia,2009-01,2026-03,15
3,petroleo,cuenca,2009-01,2025-11,17
4,petroleo,empresa,2009-01,2026-04,127
5,petroleo,provincia,2009-01,2025-12,15


In [11]:
# 4) Validación de producción negativa
negativos = df.loc[df["produccion"] < 0].copy()
negativos["magnitud_abs"] = negativos["produccion"].abs()
negativos["observacion_negativo"] = np.where(
    negativos["magnitud_abs"] < 0.001,
    "anomalia_menor_abs_lt_0_001",
    "anomalia_revisar",
)

print("=== Producción negativa ===")
print(f"Cantidad registros con produccion < 0: {len(negativos):,}")

if negativos.empty:
    print("No se detectó producción negativa.")
else:
    resumen_negativos = (
        negativos.groupby(["producto", "agrupador_tipo", "observacion_negativo"], as_index=False)
        .agg(cantidad=("produccion", "size"), produccion_min=("produccion", "min"), produccion_max=("produccion", "max"))
        .sort_values(["producto", "agrupador_tipo", "observacion_negativo"])
    )
    display(resumen_negativos)
    display(negativos.head(100))

print("Nota: no se eliminan automáticamente negativos menores; se marcan para observación.")

=== Producción negativa ===
Cantidad registros con produccion < 0: 1


,producto,agrupador_tipo,observacion_negativo,cantidad,produccion_min,produccion_max
0,petroleo,empresa,anomalia_menor_abs_lt_0_001,1,-0.000323,-0.000323


,periodo_str,periodo_dt,anio,mes,producto,agrupador_tipo,agrupador_nombre,tipo_recurso,produccion,source_resource,magnitud_abs,observacion_negativo
16290,2018-10,2018-10-01,2018,10,petroleo,empresa,PETROFARO S.A.,promedio_diario,-0.000323,Producción de petróleo promedio diaria por emp...,0.000323,anomalia_menor_abs_lt_0_001


Nota: no se eliminan automáticamente negativos menores; se marcan para observación.


In [12]:
# 5) Comparación de totales entre vistas de agregación (validación de consistencia)
totales_vista = (
    df.groupby(["producto", "periodo_str", "agrupador_tipo"], as_index=False)
    .agg(total_produccion=("produccion", "sum"))
)

resumen_vistas = (
    totales_vista.pivot_table(
        index=["producto", "periodo_str"],
        columns="agrupador_tipo",
        values="total_produccion",
        aggfunc="first",
    )
    .reset_index()
)

resumen_vistas.columns.name = None
resumen_vistas = resumen_vistas.rename(columns={
    "provincia": "total_provincia",
    "cuenca": "total_cuenca",
    "empresa": "total_empresa",
})

for col in ["total_provincia", "total_cuenca", "total_empresa"]:
    if col not in resumen_vistas.columns:
        resumen_vistas[col] = np.nan

prov = resumen_vistas["total_provincia"]
resumen_vistas["diff_cuenca_vs_provincia_pct"] = np.where(
    prov.notna() & (prov != 0),
    (resumen_vistas["total_cuenca"] - prov) / prov * 100,
    np.nan,
)
resumen_vistas["diff_empresa_vs_provincia_pct"] = np.where(
    prov.notna() & (prov != 0),
    (resumen_vistas["total_empresa"] - prov) / prov * 100,
    np.nan,
)

resumen_vistas = resumen_vistas[[
    "producto",
    "periodo_str",
    "total_provincia",
    "total_cuenca",
    "total_empresa",
    "diff_cuenca_vs_provincia_pct",
    "diff_empresa_vs_provincia_pct",
]].sort_values(["producto", "periodo_str"]).reset_index(drop=True)

resumen_vistas.to_csv(TOTALES_EXPORT_PATH, index=False)
print(f"Exportado: {TOTALES_EXPORT_PATH}")
display(resumen_vistas.head(30))

Exportado: c:\Proyectos\Dashboards\OilAndGas\exploration\data\processed\sesco_totales_por_vista_resumen.csv


,producto,periodo_str,total_provincia,total_cuenca,total_empresa,diff_cuenca_vs_provincia_pct,diff_empresa_vs_provincia_pct
0,gas,2009-01,131223.010820,131223.010820,131223.010820,2.217891e-14,4.435782e-14
1,gas,2009-02,132833.268311,132833.268311,132833.268311,2.191005e-14,0.000000e+00
2,gas,2009-03,135361.204784,135361.204784,135361.204784,-6.450260e-14,-8.600346e-14
3,gas,2009-04,135493.246151,135493.246151,135493.246151,0.000000e+00,8.591965e-14
4,gas,2009-05,135374.432953,135374.432953,135374.432953,-4.299753e-14,-2.149876e-14
5,gas,2009-06,138925.003551,138925.003551,138925.003551,-2.094931e-14,-4.189862e-14
6,gas,2009-07,138202.710745,138202.710745,138202.710745,2.105880e-14,4.211760e-14
7,gas,2009-08,129584.371960,129584.371960,129584.371960,1.122968e-13,0.000000e+00
8,gas,2009-09,133147.567907,133147.567907,133147.567907,8.743331e-14,4.371665e-14
9,gas,2009-10,132800.720557,132800.720557,132800.720557,-2.191542e-14,0.000000e+00


## Conclusiones y reglas para Streamlit

- Para totales nacionales, usar `agrupador_tipo == "provincia"`.
- Para rankings por cuenca, usar `agrupador_tipo == "cuenca"`.
- Para rankings por empresa, usar `agrupador_tipo == "empresa"`.
- Nunca sumar provincia + cuenca + empresa para obtener producción total.
- El último período válido debe calcularse según `producto + agrupador_tipo`.
- Petróleo y gas pueden tener unidades distintas; no compararlos en el mismo eje como si fueran equivalentes.
- Para comparar tendencias petróleo vs gas, usar base 100 o gráficos separados.